# LangChain y LangSmith · Herramientas y agentes

Herramientas (*tools*), llamadas a herramientas y agentes con `create_agent` de LangChain 1.x, que construye un grafo de LangGraph: el modelo decide qué herramientas llamar, el agente las ejecuta y le devuelve los resultados hasta que el modelo responde sin pedir más. También memoria con *checkpointer*, *middleware*, aprobación humana y salida estructurada.

El dominio es un asistente de viajes por Canarias. Como en el notebook anterior, el modelo es local y determinista (un `BaseChatModel` con reglas que sabe pedir herramientas), así que todo se ejecuta sin claves de API; con un modelo real el agente se crea igual, por ejemplo `create_agent("anthropic:claude-sonnet-5-5", tools=...)`.

## 1. Una herramienta

Crear con el decorador `@tool` la herramienta `tiempo(isla)` que consulte un diccionario con la previsión de cada isla. Mostrar su nombre, su descripción (sale del *docstring*), sus argumentos y el esquema JSON que recibiría el modelo. Invocarla con un diccionario de argumentos.

In [1]:
import json
import math
import re

from langchain.tools import tool

PREVISION = {
    "Tenerife": "soleado, 24 °C", "Gran Canaria": "nubes y claros, 22 °C", "Lanzarote": "ventoso, 23 °C",
    "Fuerteventura": "soleado, 25 °C", "La Palma": "lluvia débil, 18 °C", "La Gomera": "nublado, 20 °C",
    "El Hierro": "soleado, 21 °C",
}


@tool
def tiempo(isla: str) -> str:
    """Devuelve la previsión del tiempo para hoy en una isla de Canarias."""
    if isla not in PREVISION:
        raise ValueError(f"No hay previsión para '{isla}'")
    return f"{isla}: {PREVISION[isla]}"


print(tiempo.name, "|", tiempo.description)
print(tiempo.args)
print(json.dumps(tiempo.tool_call_schema.model_json_schema(), ensure_ascii=False))
print(tiempo.invoke({"isla": "La Palma"}))

tiempo | Devuelve la previsión del tiempo para hoy en una isla de Canarias.
{'isla': {'title': 'Isla', 'type': 'string'}}
{"description": "Devuelve la previsión del tiempo para hoy en una isla de Canarias.", "properties": {"isla": {"title": "Isla", "type": "string"}}, "required": ["isla"], "title": "tiempo", "type": "object"}
La Palma: lluvia débil, 18 °C


## 2. Esquemas de argumentos

Crear dos herramientas más con argumentos validados:
- `distancia(origen, destino)`, con un `args_schema` de Pydantic donde las islas son un `Literal` y cada campo tiene descripción; calcula la distancia ortodrómica (fórmula del semiverseno) entre las capitales.
- `convertir_moneda(cantidad, de, a)`, que toma las descripciones de los argumentos de un *docstring* de estilo Google con `parse_docstring=True`.

Mostrar sus esquemas y comprobar que unos argumentos inválidos lanzan un error de validación (capturarlo e imprimirlo).

In [2]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

COORDENADAS = {
    "Tenerife": (28.4636, -16.2518), "Gran Canaria": (28.1235, -15.4363), "Lanzarote": (28.9630, -13.5477),
    "Fuerteventura": (28.5004, -13.8627), "La Palma": (28.6835, -17.7642), "La Gomera": (28.0916, -17.1133),
    "El Hierro": (27.8064, -17.9158),
}
Isla = Literal["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera", "El Hierro"]


class ArgsDistancia(BaseModel):
    origen: Isla = Field(description="Isla de origen")
    destino: Isla = Field(description="Isla de destino")


@tool(args_schema=ArgsDistancia)
def distancia(origen: str, destino: str) -> float:
    """Distancia en línea recta, en km, entre las capitales de dos islas."""
    (la1, lo1), (la2, lo2) = (map(math.radians, COORDENADAS[i]) for i in (origen, destino))
    h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return round(2 * 6371 * math.asin(math.sqrt(h)), 1)


CAMBIO_DESDE_EURO = {"EUR": 1.0, "USD": 1.08, "GBP": 0.85}


@tool(parse_docstring=True)
def convertir_moneda(cantidad: float, de: str, a: str) -> float:
    """Convierte una cantidad de dinero entre monedas.

    Args:
        cantidad: Importe que se quiere convertir.
        de: Código ISO de la moneda de origen (EUR, USD o GBP).
        a: Código ISO de la moneda de destino.
    """
    return round(cantidad / CAMBIO_DESDE_EURO[de] * CAMBIO_DESDE_EURO[a], 2)


for herramienta in (distancia, convertir_moneda):
    print(herramienta.name, json.dumps(herramienta.tool_call_schema.model_json_schema()["properties"], ensure_ascii=False))
print(distancia.invoke({"origen": "Gran Canaria", "destino": "Tenerife"}), "km")
print(convertir_moneda.invoke({"cantidad": 100, "de": "EUR", "a": "USD"}), "USD")

try:
    distancia.invoke({"origen": "Mallorca", "destino": "Tenerife"})
except ValidationError as e:
    print("Error de validación:", e.errors()[0]["msg"])

distancia {"origen": {"description": "Isla de origen", "enum": ["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera", "El Hierro"], "title": "Origen", "type": "string"}, "destino": {"description": "Isla de destino", "enum": ["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera", "El Hierro"], "title": "Destino", "type": "string"}}
convertir_moneda {"cantidad": {"description": "Importe que se quiere convertir.", "title": "Cantidad", "type": "number"}, "de": {"description": "Código ISO de la moneda de origen (EUR, USD o GBP).", "title": "De", "type": "string"}, "a": {"description": "Código ISO de la moneda de destino.", "title": "A", "type": "string"}}
88.3 km
108.0 USD
Error de validación: Input should be 'Tenerife', 'Gran Canaria', 'Lanzarote', 'Fuerteventura', 'La Palma', 'La Gomera' or 'El Hierro'


## 3. Contenido y artefacto

Crear `buscar_vuelos(origen, destino)` con `response_format="content_and_artifact"`: devuelve un texto breve para el modelo y, aparte, la lista completa de vuelos como artefacto. Invocarla con una llamada a herramienta (`ToolCall`) y comprobar que el `ToolMessage` resultante guarda el texto en `content` y la lista en `artifact`.

In [3]:
from langchain.messages import AIMessage, HumanMessage, ToolCall, ToolMessage


@tool(response_format="content_and_artifact")
def buscar_vuelos(origen: str, destino: str) -> tuple[str, list[dict]]:
    """Busca los vuelos de hoy entre dos islas."""
    vuelos = [
        {"vuelo": f"NT{100 + i}", "salida": f"{7 + 3 * i:02d}:15", "precio": 45 + 12 * i}
        for i in range(4)
    ]
    barato = min(vuelos, key=lambda v: v["precio"])
    return f"{len(vuelos)} vuelos de {origen} a {destino}; el más barato sale a las {barato['salida']} por {barato['precio']} €", vuelos


mensaje = buscar_vuelos.invoke(ToolCall(name="buscar_vuelos", args={"origen": "Gran Canaria", "destino": "La Palma"}, id="llamada_1", type="tool_call"))
print(type(mensaje).__name__, "|", mensaje.content)
print(mensaje.artifact)
assert isinstance(mensaje, ToolMessage) and len(mensaje.artifact) == 4

ToolMessage | 4 vuelos de Gran Canaria a La Palma; el más barato sale a las 07:15 por 45 €
[{'vuelo': 'NT100', 'salida': '07:15', 'precio': 45}, {'vuelo': 'NT101', 'salida': '10:15', 'precio': 57}, {'vuelo': 'NT102', 'salida': '13:15', 'precio': 69}, {'vuelo': 'NT103', 'salida': '16:15', 'precio': 81}]


## 4. Ejecutar llamadas a herramientas

Dado un `AIMessage` con dos llamadas a herramientas (como el que devolvería un modelo), ejecutar cada una buscando la herramienta por su nombre y construir los `ToolMessage` de respuesta, enlazados con el `tool_call_id`. Mostrar la conversación resultante.

In [4]:
herramientas = [tiempo, distancia, convertir_moneda, buscar_vuelos]
por_nombre = {h.name: h for h in herramientas}

peticion = AIMessage(content="", tool_calls=[
    {"name": "tiempo", "args": {"isla": "El Hierro"}, "id": "c1"},
    {"name": "convertir_moneda", "args": {"cantidad": 50, "de": "GBP", "a": "EUR"}, "id": "c2"},
])
respuestas = [por_nombre[llamada["name"]].invoke(llamada) for llamada in peticion.tool_calls]
for m in [peticion, *respuestas]:
    m.pretty_print()
assert [r.tool_call_id for r in respuestas] == ["c1", "c2"]

================================== Ai Message ==================================
Tool Calls:
  tiempo (c1)
 Call ID: c1
  Args:
    isla: El Hierro
  convertir_moneda (c2)
 Call ID: c2
  Args:
    cantidad: 50
    de: GBP
    a: EUR
================================= Tool Message =================================
Name: tiempo

El Hierro: soleado, 21 °C
================================= Tool Message =================================
Name: convertir_moneda

58.82


## 5. Un modelo que llama herramientas

Crear `ModeloViajes`, un `BaseChatModel` determinista que imita a un modelo con *tool calling*:
- `bind_tools` devuelve una copia que conoce los nombres de las herramientas disponibles.
- Si el último mensaje es del usuario, busca intenciones con expresiones regulares (tiempo en una isla, distancia entre dos islas, conversión de moneda, reserva de hotel) y pide en paralelo las herramientas que estén disponibles.
- Si el último mensaje es el resultado de una herramienta, redacta la respuesta final con los resultados recibidos desde su última petición.

Comprobar que una pregunta con dos intenciones produce dos llamadas.

In [5]:
from langchain_core.language_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult

ISLAS = "|".join(COORDENADAS)
INTENCIONES = [
    ("tiempo", rf"tiempo (?:hace )?en (?P<isla>{ISLAS})", lambda m: {"isla": m["isla"]}),
    ("distancia", rf"(?:distancia|lejos).*?(?P<origen>{ISLAS}) (?:y|a) (?P<destino>{ISLAS})", lambda m: m.groupdict()),
    ("convertir_moneda", r"(?P<cantidad>\d+(?:\.\d+)?) (?P<de>EUR|USD|GBP) (?:a|en) (?P<a>EUR|USD|GBP)",
     lambda m: {"cantidad": float(m["cantidad"]), "de": m["de"], "a": m["a"]}),
    ("reservar_hotel", rf"reserva.*hotel en (?P<isla>{ISLAS}).*?(?P<noches>\d+) noches?",
     lambda m: {"isla": m["isla"], "noches": int(m["noches"])}),
]


class ModeloViajes(BaseChatModel):
    disponibles: list[str] = []

    @property
    def _llm_type(self) -> str:
        return "viajes"

    def bind_tools(self, tools, **kwargs):
        return self.model_copy(update={"disponibles": [getattr(t, "name", None) or t.__name__ for t in tools]})

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        ultimo = messages[-1]
        if isinstance(ultimo, ToolMessage):
            resultados = []
            for m in reversed(messages):
                if not isinstance(m, ToolMessage):
                    break
                resultados.append(m.text)
            respuesta = AIMessage("Esto es lo que he averiguado: " + "; ".join(r.rstrip(".") for r in reversed(resultados)) + ".")
        else:
            llamadas = [
                {"name": nombre, "args": argumentos(m), "id": f"llamada_{len(messages)}_{i}"}
                for i, (nombre, patron, argumentos) in enumerate(INTENCIONES)
                if nombre in self.disponibles and (m := re.search(patron, ultimo.text, re.IGNORECASE))
            ]
            respuesta = AIMessage("", tool_calls=llamadas) if llamadas else AIMessage("Solo sé del tiempo, distancias, monedas y hoteles en Canarias.")
        return ChatResult(generations=[ChatGeneration(message=respuesta)])


modelo = ModeloViajes().bind_tools(herramientas)
pregunta = HumanMessage("¿Qué tiempo hace en Lanzarote? ¿Y a qué distancia están Tenerife y La Gomera?")
respuesta = modelo.invoke([pregunta])
print(respuesta.tool_calls)
assert [c["name"] for c in respuesta.tool_calls] == ["tiempo", "distancia"]
print(ModeloViajes().invoke("¿Qué tiempo hace en Lanzarote?").content)

[{'name': 'tiempo', 'args': {'isla': 'Lanzarote'}, 'id': 'llamada_1_0', 'type': 'tool_call'}, {'name': 'distancia', 'args': {'origen': 'Tenerife', 'destino': 'La Gomera'}, 'id': 'llamada_1_1', 'type': 'tool_call'}]
Solo sé del tiempo, distancias, monedas y hoteles en Canarias.


## 6. El bucle del agente a mano

Programar el bucle de un agente: llamar al modelo, ejecutar las herramientas que pida, añadir los resultados a los mensajes y repetir hasta que responda sin llamadas (con un máximo de iteraciones). Mostrar cada paso y la respuesta final.

In [6]:
def agente_a_mano(pregunta, modelo, herramientas, max_pasos=5):
    por_nombre = {h.name: h for h in herramientas}
    mensajes = [HumanMessage(pregunta)]
    for paso in range(1, max_pasos + 1):
        respuesta = modelo.invoke(mensajes)
        mensajes.append(respuesta)
        if not respuesta.tool_calls:
            return mensajes
        print(f"Paso {paso}: el modelo pide {[(c['name'], c['args']) for c in respuesta.tool_calls]}")
        mensajes += [por_nombre[c["name"]].invoke(c) for c in respuesta.tool_calls]
    raise RuntimeError("Demasiados pasos")


mensajes = agente_a_mano("¿Qué tiempo hace en Fuerteventura? Y cambia 200 EUR a GBP", modelo, herramientas)
print("Respuesta:", mensajes[-1].content)

Paso 1: el modelo pide [('tiempo', {'isla': 'Fuerteventura'}), ('convertir_moneda', {'cantidad': 200.0, 'de': 'EUR', 'a': 'GBP'})]
Respuesta: Esto es lo que he averiguado: Fuerteventura: soleado, 25 °C; 170.0.


## 7. Un agente con `create_agent`

Crear el mismo agente con `create_agent`, pasándole el modelo, las herramientas y un `system_prompt`. Invocarlo, mostrar todos los mensajes del estado final y comprobar que la respuesta coincide con la del bucle a mano.

In [7]:
from langchain.agents import create_agent

agente = create_agent(ModeloViajes(), tools=herramientas, system_prompt="Eres un asistente de viajes por Canarias.")
estado = agente.invoke({"messages": [{"role": "user", "content": "¿Qué tiempo hace en Fuerteventura? Y cambia 200 EUR a GBP"}]})
for m in estado["messages"]:
    m.pretty_print()
assert estado["messages"][-1].content == mensajes[-1].content

================================ Human Message =================================

¿Qué tiempo hace en Fuerteventura? Y cambia 200 EUR a GBP
================================== Ai Message ==================================
Tool Calls:
  tiempo (llamada_2_0)
 Call ID: llamada_2_0
  Args:
    isla: Fuerteventura
  convertir_moneda (llamada_2_2)
 Call ID: llamada_2_2
  Args:
    cantidad: 200.0
    de: EUR
    a: GBP
================================= Tool Message =================================
Name: tiempo

Fuerteventura: soleado, 25 °C
================================= Tool Message =================================
Name: convertir_moneda

170.0
================================== Ai Message ==================================

Esto es lo que he averiguado: Fuerteventura: soleado, 25 °C; 170.0.


## 8. Ver el agente paso a paso

Recorrer la ejecución del agente con `stream(..., stream_mode="updates")` y mostrar, para cada actualización, el nodo del grafo que la produce (`model` o `tools`) y un resumen de los mensajes que añade. Mostrar también los nodos del grafo.

In [8]:
print(list(agente.get_graph().nodes))
for actualizacion in agente.stream(
    {"messages": [{"role": "user", "content": "¿A qué distancia están El Hierro y Lanzarote? ¿Qué tiempo hace en El Hierro?"}]},
    stream_mode="updates",
):
    for nodo, cambios in actualizacion.items():
        for m in cambios["messages"]:
            resumen = [c["name"] for c in m.tool_calls] if getattr(m, "tool_calls", None) else m.text
            print(f"{nodo:>6} | {type(m).__name__:<11} | {resumen}")

['__start__', 'model', 'tools', '__end__']
 model | AIMessage   | ['tiempo', 'distancia']
 tools | ToolMessage | El Hierro: soleado, 21 °C
 tools | ToolMessage | 446.2
 model | AIMessage   | Esto es lo que he averiguado: El Hierro: soleado, 21 °C; 446.2.


## 9. Memoria entre turnos

Crear el agente con un *checkpointer* `InMemorySaver` e invocarlo dos veces con el mismo `thread_id` y una vez con otro distinto. Comprobar con `get_state` que el primer hilo acumula los mensajes de los dos turnos y que el segundo empieza vacío.

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

agente_memoria = create_agent(ModeloViajes(), tools=herramientas, checkpointer=InMemorySaver())
hilo_a = {"configurable": {"thread_id": "a"}}
hilo_b = {"configurable": {"thread_id": "b"}}

agente_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué tiempo hace en Tenerife?"}]}, hilo_a)
agente_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué tiempo hace en La Gomera?"}]}, hilo_a)
agente_memoria.invoke({"messages": [{"role": "user", "content": "Hola"}]}, hilo_b)

for nombre, hilo in [("a", hilo_a), ("b", hilo_b)]:
    mensajes_hilo = agente_memoria.get_state(hilo).values["messages"]
    print(f"Hilo {nombre}: {len(mensajes_hilo)} mensajes ->", [type(m).__name__ for m in mensajes_hilo])
assert len(agente_memoria.get_state(hilo_a).values["messages"]) == 8

Hilo a: 8 mensajes -> ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage', 'HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']
Hilo b: 2 mensajes -> ['HumanMessage', 'AIMessage']


## 10. Middleware

Añadir al agente tres *middlewares*:
- `@wrap_tool_call` que registre cada llamada y, si la herramienta lanza una excepción, devuelva al modelo un `ToolMessage` con el error en lugar de romper el agente.
- `@before_model` y `@after_model` con un estado propio que cuente las llamadas al modelo y corte la ejecución (`jump_to: "end"`) si se superan 3.

Probarlo con una pregunta cuya herramienta falla (una isla sin previsión).

In [10]:
from typing import Any

from langchain.agents.middleware import AgentState, after_model, before_model, wrap_tool_call
from typing_extensions import NotRequired

copia_prevision = dict(PREVISION)
del PREVISION["La Gomera"]


@wrap_tool_call
def registrar_y_proteger(request, handler):
    llamada = request.tool_call
    print(f"  -> {llamada['name']}({llamada['args']})")
    try:
        return handler(request)
    except Exception as e:
        print(f"  !! {type(e).__name__}: {e}")
        return ToolMessage(f"Error: {e}", tool_call_id=llamada["id"], status="error")


class EstadoContador(AgentState):
    llamadas_modelo: NotRequired[int]


@before_model(state_schema=EstadoContador, can_jump_to=["end"])
def limitar(state, runtime) -> dict[str, Any] | None:
    if state.get("llamadas_modelo", 0) >= 3:
        return {"jump_to": "end"}
    return None


@after_model(state_schema=EstadoContador)
def contar(state, runtime) -> dict[str, Any] | None:
    return {"llamadas_modelo": state.get("llamadas_modelo", 0) + 1}


agente_mw = create_agent(ModeloViajes(), tools=herramientas, middleware=[registrar_y_proteger, limitar, contar])
estado = agente_mw.invoke({"messages": [{"role": "user", "content": "¿Qué tiempo hace en La Gomera? ¿Y a qué distancia están La Gomera y Tenerife?"}]})
print(estado["messages"][-1].content)
print("Llamadas al modelo:", estado["llamadas_modelo"])
PREVISION.update(copia_prevision)

  -> tiempo({'isla': 'La Gomera'})
  !! ValueError: No hay previsión para 'La Gomera'
  -> distancia({'origen': 'La Gomera', 'destino': 'Tenerife'})
Esto es lo que he averiguado: Error: No hay previsión para 'La Gomera'; 94.0.
Llamadas al modelo: 2


## 11. Aprobación humana

Añadir una herramienta `reservar_hotel(isla, noches)` y crear el agente con `HumanInTheLoopMiddleware` para que se detenga antes de ejecutarla (las demás herramientas no necesitan aprobación). Mostrar la petición de aprobación que devuelve la interrupción y reanudar con `Command(resume=...)`: en un hilo aprobar la reserva y en otro rechazarla con un mensaje.

In [11]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

reservas = []


@tool
def reservar_hotel(isla: str, noches: int) -> str:
    """Reserva un hotel en una isla durante varias noches."""
    reservas.append((isla, noches))
    return f"Reserva confirmada: {noches} noches en {isla}, localizador H{len(reservas):03d}"


agente_hitl = create_agent(
    ModeloViajes(),
    tools=[*herramientas, reservar_hotel],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"reservar_hotel": True})],
    checkpointer=InMemorySaver(),
)
peticion = {"messages": [{"role": "user", "content": "Reserva un hotel en La Palma para 3 noches"}]}
decisiones = {"aprobar": {"type": "approve"}, "rechazar": {"type": "reject", "message": "El usuario prefiere esperar."}}

for hilo, decision in decisiones.items():
    config = {"configurable": {"thread_id": hilo}}
    pausado = agente_hitl.invoke(peticion, config)
    print(f"[{hilo}] interrupción:", pausado["__interrupt__"][0].value["action_requests"])
    final = agente_hitl.invoke(Command(resume={"decisions": [decision]}), config)
    print(f"[{hilo}] {final['messages'][-1].content}")
assert reservas == [("La Palma", 3)]

[aprobar] interrupción: [{'name': 'reservar_hotel', 'args': {'isla': 'La Palma', 'noches': 3}, 'description': "Tool execution requires approval\n\nTool: reservar_hotel\nArgs: {'isla': 'La Palma', 'noches': 3}"}]
[aprobar] Esto es lo que he averiguado: Reserva confirmada: 3 noches en La Palma, localizador H001.
[rechazar] interrupción: [{'name': 'reservar_hotel', 'args': {'isla': 'La Palma', 'noches': 3}, 'description': "Tool execution requires approval\n\nTool: reservar_hotel\nArgs: {'isla': 'La Palma', 'noches': 3}"}]
[rechazar] Esto es lo que he averiguado: User rejected the tool call for `reservar_hotel` with reason: El usuario prefiere esperar.


## 12. Salida estructurada

Pedir al agente una respuesta estructurada con `response_format=ToolStrategy(PlanViaje)`, donde `PlanViaje` es un modelo de Pydantic. Con la estrategia de herramientas, el modelo responde llamando a una herramienta con el nombre del esquema; simularlo con un `GenericFakeChatModel` con guion (primero pide el tiempo y luego entrega el plan). Comprobar que `structured_response` es un `PlanViaje` válido.

In [12]:
from langchain.agents.structured_output import ToolStrategy
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


class PlanViaje(BaseModel):
    isla: Isla
    dias: int = Field(gt=0)
    actividades: list[str]


class ModeloGuion(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


guion = ModeloGuion(messages=iter([
    AIMessage("", tool_calls=[{"name": "tiempo", "args": {"isla": "La Palma"}, "id": "t1"}]),
    AIMessage("", tool_calls=[{"name": "PlanViaje", "args": {"isla": "La Palma", "dias": 3, "actividades": ["Roque de los Muchachos", "Caldera de Taburiente", "Observar estrellas"]}, "id": "t2"}]),
]))
agente_plan = create_agent(guion, tools=[tiempo], response_format=ToolStrategy(PlanViaje))
estado = agente_plan.invoke({"messages": [{"role": "user", "content": "Planifica 3 días en La Palma"}]})
plan = estado["structured_response"]
print(repr(plan))
print([type(m).__name__ for m in estado["messages"]])
assert isinstance(plan, PlanViaje) and plan.dias == 3

PlanViaje(isla='La Palma', dias=3, actividades=['Roque de los Muchachos', 'Caldera de Taburiente', 'Observar estrellas'])
['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage', 'ToolMessage']


## 13. Probar la trayectoria del agente

Escribir una batería de casos `(pregunta, herramientas esperadas)` y comprobar, para cada uno, que el agente llama exactamente a esas herramientas y termina con una respuesta sin llamadas pendientes. Mostrar una tabla con el resultado de cada caso.

In [13]:
casos = [
    ("¿Qué tiempo hace en Gran Canaria?", ["tiempo"]),
    ("¿A qué distancia están Lanzarote y Fuerteventura?", ["distancia"]),
    ("Convierte 30 USD a EUR y dime el tiempo en El Hierro", ["tiempo", "convertir_moneda"]),
    ("Cuéntame un chiste", []),
]


def trayectoria(estado):
    return [c["name"] for m in estado["messages"] if isinstance(m, AIMessage) for c in m.tool_calls]


for pregunta, esperadas in casos:
    estado = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})
    obtenidas = trayectoria(estado)
    ok = obtenidas == esperadas and not estado["messages"][-1].tool_calls
    print(f"{'OK ' if ok else 'MAL'} {pregunta:<55} {obtenidas}")
    assert ok

OK  ¿Qué tiempo hace en Gran Canaria?                       ['tiempo']
OK  ¿A qué distancia están Lanzarote y Fuerteventura?       ['distancia']
OK  Convierte 30 USD a EUR y dime el tiempo en El Hierro    ['tiempo', 'convertir_moneda']
OK  Cuéntame un chiste                                      []
